# Multiplex IMC Image sorting pipeline

Each image from the same sample, after training and predictions of IMC_Denoised models, were saved onto a new Denoised Image folder separated by isotope identifier used.

With this pipeline it was possible to ensure each set of Multiplex Images were reorganised back together into a folder with the sample ID, to later on merge all 37 channels in a multi layered image.

This pipeline recursively scans through folders, detecting sample names and channel identifiers while also checking for the presence of a total of 37 channels within the each final sample folder.

## Identifier-Based TIFF Image Grouping

In [2]:
import os
import shutil
from pathlib import Path

def extract_identifier(filename: str) -> str:
    parts = filename.split("_")
    if len(parts) > 1:
        return parts[1]
    return "unknown"

def group_images_by_identifier(source_dir: str, dest_dir: str, move: bool = False):
    """
    Groups TIFF images by extracted sample identifier and organizes them into folders.

    Args:
        source_dir (str): The parent directory containing images (may include subfolders).
        dest_dir (str): The directory where grouped folders will be created.
        move (bool): If True, files will be moved. If False, files will be copied.
    """
    source_path = Path(source_dir)
    dest_path = Path(dest_dir)
    dest_path.mkdir(parents=True, exist_ok=True)

    for root, _, files in os.walk(source_path):
        for file in files:
            if file.endswith(('.tif', '.tiff')):
                identifier = extract_identifier(file)
                new_folder = dest_path / identifier
                new_folder.mkdir(exist_ok=True)

                src_file = Path(root) / file
                dst_file = new_folder / file

                if move:
                    shutil.move(str(src_file), str(dst_file))
                else:
                    shutil.copy2(str(src_file), str(dst_file))

    print(f"Done. Files grouped in: {dest_path}")
    
group_images_by_identifier(
    source_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/TrialDataset",
    dest_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial",
    move=False  # Change to True if you want to move instead of copy
)

Done. Files grouped in: /Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial


## Hierarchical Image Grouping by ROI and Sample Identifier

In [3]:
import os
import shutil
from pathlib import Path
import re

def extract_roi_and_identifier(filename: str):
    """
    Extracts the ROI and sample identifier (e.g., Piombetti) from filename.
    Assumes format like 'ROI045_Piombetti ...'
    """
    match = re.match(r"(ROI\d+)_([^\s_]+)", filename)
    if match:
        roi = match.group(1)
        identifier = match.group(2)
        return roi, identifier
    return None, None

def group_images_by_identifier_and_roi(source_dir: str, dest_dir: str, move: bool = False):
    """
    Groups images by sample identifier, then by ROI.
    
    Args:
        source_dir (str): Root directory containing channel folders.
        dest_dir (str): Destination directory to group by identifier/ROI.
        move (bool): Whether to move (True) or copy (False) files.
    """
    source_path = Path(source_dir)
    dest_path = Path(dest_dir)
    dest_path.mkdir(parents=True, exist_ok=True)

    for root, _, files in os.walk(source_path):
        for file in files:
            if file.endswith(('.tif', '.tiff')):
                roi, identifier = extract_roi_and_identifier(file)
                if not roi or not identifier:
                    print(f"Skipping unrecognized file: {file}")
                    continue

                # Build destination folder: /dest/Identifier/ROI###/
                new_folder = dest_path / identifier / roi
                new_folder.mkdir(parents=True, exist_ok=True)

                src_file = Path(root) / file
                dst_file = new_folder / file

                if move:
                    shutil.move(str(src_file), str(dst_file))
                else:
                    shutil.copy2(str(src_file), str(dst_file))

    print(f"Done. Images grouped under: {dest_path}")

group_images_by_identifier_and_roi(
    source_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/TrialDataset",
    dest_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial",
    move=False
)

Skipping unrecognized file: ROI012_ Fanesi 91-IST-3669 tum.periferico_149Sm_CD11b.ome_denoised.ome.tiff
Skipping unrecognized file: ROI012_ Fanesi 91-IST-3669 tum.periferico_147Sm_CD163+.ome_denoised.ome.tiff
Skipping unrecognized file: ROI012_ Fanesi 91-IST-3669 tum.periferico_169Tm_Collagen-I.ome_denoised.ome.tiff
Done. Images grouped under: /Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial


## Enhanced ROI and Identifier Parsing with Flexible Delimiters

In [5]:
import re

def extract_roi_and_identifier(filename: str):
    """
    Extracts ROI (e.g., 'ROI012') and identifier (e.g., 'Fanesi') from filename.
    Handles cases like 'ROI012_Fanesi' or 'ROI012_ Fanesi'.
    """
    # Allow spaces and/or underscores between ROI and the identifier
    match = re.match(r"(ROI\d+)[\s_]+([^\s_]+)", filename)
    if match:
        roi = match.group(1)
        identifier = match.group(2)
        return roi, identifier
    return None, None

group_images_by_identifier_and_roi(
    source_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/TrialDataset",
    dest_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial",
    move=False
)


Done. Images grouped under: /Users/hugoroca/Desktop/QMUL/CAN7036/GroupedImagesTrial


## Robust ROI-Based Grouping with Error Handling and File Validation

In [5]:
import os
import re
import shutil
from pathlib import Path
from typing import Optional


def extract_identifier(filename: str) -> Optional[str]:
    """
    Return everything from the start of the filename (ROI### …)
    up to—but NOT including—the underscore that precedes the
    isotope tag (e.g. _149Sm, _147Sm, _169Tm).

    Works with spaces, multiple underscores, etc.
    """
    # ^ start  (.*?)  look‑ahead for _[digits][element]
    pattern = r"^(.*?)(?=_\d{2,3}[A-Z][a-z]?)"
    m = re.match(pattern, filename)
    return m.group(1).strip() if m else None


def group_images_by_identifier(
    source_dir: str,
    dest_dir: str,
    move: bool = False,
    valid_ext: tuple = (".tif", ".tiff")
) -> None:
    """
    Walk through `source_dir`, copy/move each image whose name matches
    the ROI pattern into `dest_dir/<identifier>/`.

    Parameters
    ----------
    source_dir : str
        Root directory containing any depth of sub‑folders.
    dest_dir   : str
        Output root where grouped folders will be created.
    move       : bool
        If True, files are moved; if False (default), they are copied.
    valid_ext  : tuple
        Recognised image extensions.
    """
    src_root = Path(source_dir)
    dst_root = Path(dest_dir)
    dst_root.mkdir(parents=True, exist_ok=True)

    for root, _, files in os.walk(src_root):
        for fname in files:
            if not fname.lower().endswith(valid_ext):
                continue

            identifier = extract_identifier(fname)
            if identifier is None:
                print(f"⚠️  Skipping unrecognised file: {fname}")
                continue

            dst_dir = dst_root / identifier
            dst_dir.mkdir(parents=True, exist_ok=True)

            src_file = Path(root) / fname
            dst_file = dst_dir / fname

            try:
                if move:
                    shutil.move(src_file, dst_file)
                else:
                    shutil.copy2(src_file, dst_file)
            except shutil.SameFileError:
                print(f"ℹ️  Duplicate ignored: {dst_file.name}")

    print(f"✅ Done. Images grouped under: {dst_root.resolve()}")


# ----------------- example call -----------------
group_images_by_identifier(
source_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/Denoised_Images",
dest_dir="/Users/hugoroca/Desktop/QMUL/CAN7036/Grouped_Denoised_Images",
move=False)    # set True if you prefer moving


✅ Done. Images grouped under: /Users/hugoroca/Desktop/QMUL/CAN7036/Grouped_Denoised_Images


## Quality Control: Folder-Level Image Count Verification for Grouped Datasets

In [12]:
from pathlib import Path
from typing import Tuple, Dict

def check_image_counts(
    root_dir: str,
    expected: int = 37,
    valid_ext: Tuple[str, ...] = (".tif", ".tiff"),
) -> Dict[str, int]:
    """
    Walk through each first‑level folder in `root_dir` and count the number
    of image files it contains (recursively).  Prints a summary and returns
    a {folder_name: image_count} dictionary.

    Parameters
    ----------
    root_dir  : str
        The destination directory created by the grouping script.
    expected  : int, default 15
        The target number of images each folder should have.
    valid_ext : tuple[str], default ('.tif', '.tiff')
        File extensions to count as images.

    Returns
    -------
    Dict[str, int]
        Mapping of folder name → number of images found.
    """
    root = Path(root_dir)
    results: Dict[str, int] = {}

    for folder in root.iterdir():
        if not folder.is_dir():
            continue

        count = sum(
            1
            for f in folder.rglob("*")
            if f.is_file() and f.suffix.lower() in valid_ext
        )
        results[folder.name] = count

        status = "✓ OK" if count == expected else "⚠️  MISMATCH"
        print(f"{folder.name:<40} {count:>3} images  {status}")

    return results


# ----------------- example call -----------------
check_image_counts("/Users/hugoroca/Desktop/QMUL/CAN7036/Dataset_Verona-32bit")


ROI045_PDAC15- 08106 periferic tum        37 images  ✓ OK
ROI045_Piombetti 03-IST-3305 tum.periferico  37 images  ✓ OK
ROI004_lervese 89-IST-10634 panc. Vicino  37 images  ✓ OK
ROI021_IPMN3_2823_A4                      37 images  ✓ OK
ROI003_Pizziferri 04-IST-5061 tum.periferico  37 images  ✓ OK
ROI035_IPMN1_2478_A10                     37 images  ✓ OK
ROI048_IPMN3_4330_A15                     37 images  ✓ OK
ROI023_Pica 03-IST-12823 tum.periferico   37 images  ✓ OK
ROI003_IPMN3_1836_A11                     37 images  ✓ OK
ROI008_Ginesi  04-IST-2955 tum.periferico  37 images  ✓ OK
ROI045_Viglienzone 94-IST- 8752A  tum.centrale  37 images  ✓ OK
ROI011_Moriconi 04-IST-2695 tum.periferico  37 images  ✓ OK
ROI057_PDAC15-06906 periferic tum         37 images  ✓ OK
ROI017_IPMN2_3939_A9                      37 images  ✓ OK
ROI018_Sandri 04-IST-1087 tum.periferico  37 images  ✓ OK
ROI053_IPMN2_4992_A4                      37 images  ✓ OK
ROI018_PDAC15-10911 near panc             37 images  ✓ O

ROI044_IPMN3_3432_A17                     37 images  ✓ OK
ROI059_IPMN1_1008_A15                     37 images  ✓ OK
ROI017_IPMN1_2465_A18                     37 images  ✓ OK
ROI048_D_Attimis 94-IST-106A1  tum.centrale  37 images  ✓ OK
ROI048_PDAC15- 07909 tumour               37 images  ✓ OK
ROI042_IPMN1_2593_A29                     37 images  ✓ OK
ROI014_Pizziferri 04-IST-5061 tum.centrale  37 images  ✓ OK
ROI004_IPMN1_3159_A2                      37 images  ✓ OK
ROI011_Fanesi 91-IST-3669 tum.periferico  37 images  ✓ OK
ROI026_Morandini  03-IST-5931  tum.centrale  37 images  ✓ OK
ROI060_PDAC15-06147 near panc             37 images  ✓ OK
ROI051_IPMN3_4904_A7                      37 images  ✓ OK
ROI040_PDAC15-09109 near panc             37 images  ✓ OK
ROI026_IPMN3_3025_A20                     37 images  ✓ OK
ROI026_Cavazza 92-IST-3924A  tum.centrale  37 images  ✓ OK
ROI018_IPMN1_2465_A18                     37 images  ✓ OK
ROI041_Violante 03-IST-10177 pancr.vicino  37 images  ✓ OK
ROI0

ROI018_Paolini 91-IST-11569 tumore        37 images  ✓ OK
ROI043_Legnani 93-IST-9355C pancr.lontano  37 images  ✓ OK
ROI043_Piombetti 03-IST-3305 tum.centrale  37 images  ✓ OK
ROI039_Violante 03-IST-10177  tumore      37 images  ✓ OK
ROI055_IPMN1_5516_A15                     37 images  ✓ OK
ROI019_IPMN3_2823_A4                      37 images  ✓ OK
ROI012_IPMN3_2611_A7                      37 images  ✓ OK
ROI036_Facchin 03-IST-7123   pancr.lontano  37 images  ✓ OK
ROI016_IPMN2_3939_A9                      37 images  ✓ OK
ROI064_Mazzarini 03-IST-5656 pancr.lontano  37 images  ✓ OK
ROI032_PDAC15T-4190  tumour               37 images  ✓ OK
ROI020_IPMN3_2823_A4                      37 images  ✓ OK
ROI014_IPMN3_2614_A16                     37 images  ✓ OK
ROI038_IPMN2_5394_A9                      37 images  ✓ OK
ROI049_Viglienzone 94-IST-10007 panc.lontano  37 images  ✓ OK
ROI027_Morandini 03-IST-5931 tum.periferico  37 images  ✓ OK
ROI037_IPMN2_5474_A9                      37 images  ✓ OK
R

{'ROI045_PDAC15- 08106 periferic tum': 37,
 'ROI045_Piombetti 03-IST-3305 tum.periferico': 37,
 'ROI004_lervese 89-IST-10634 panc. Vicino': 37,
 'ROI021_IPMN3_2823_A4': 37,
 'ROI003_Pizziferri 04-IST-5061 tum.periferico': 37,
 'ROI035_IPMN1_2478_A10': 37,
 'ROI048_IPMN3_4330_A15': 37,
 'ROI023_Pica 03-IST-12823 tum.periferico': 37,
 'ROI003_IPMN3_1836_A11': 37,
 'ROI008_Ginesi  04-IST-2955 tum.periferico': 37,
 'ROI045_Viglienzone 94-IST- 8752A  tum.centrale': 37,
 'ROI011_Moriconi 04-IST-2695 tum.periferico': 37,
 'ROI057_PDAC15-06906 periferic tum': 37,
 'ROI017_IPMN2_3939_A9': 37,
 'ROI018_Sandri 04-IST-1087 tum.periferico': 37,
 'ROI053_IPMN2_4992_A4': 37,
 'ROI018_PDAC15-10911 near panc': 37,
 'ROI049_IPMN2_5040_A16': 37,
 'ROI014_Pegorari 90-IST-4198 pancr.vicino': 37,
 'ROI043_IPMN1_2588_A2': 37,
 'ROI049_Facchin 03-IST-7123  tum.centrale': 37,
 'ROI006_IPMN2_4096_A16': 37,
 'ROI008_IPMN3_2490_A3': 37,
 'ROI046_IPMN3_3432_A17': 37,
 'ROI051_Iacono 03-IST-12273 tumore': 37,
 'ROI